# AegisEye — Model A v2: Accident Detector (Binary)

**Goal:** Detect accident vs noaccident. That's it. No severity, no vehicles — those are downstream.

**Datasets merged:**
- Severity Dataset (Roboflow, 28,135 train) — remapped from 6 classes to 2
- NTA Accident (Roboflow, ~6,803 train) — already binary, just needs class ID alignment

**Final classes:** `0: accident`, `1: noaccident`

**Estimated time:** ~6.5 hours on dual T4 (fits in one 9hr Kaggle session)

In [1]:
# ============================================
# CELL 0 — Install + GPU check
# STOP if you don't see 2x Tesla T4
# ============================================
!pip install ultralytics roboflow pyyaml --quiet
import torch, os, glob, shutil, collections, random, yaml
random.seed(42)

print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

assert torch.cuda.device_count() >= 1, "❌ No GPU — go to Settings → Accelerator → GPU T4 x2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 93.7 MB/s eta 0:00:00
GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [2]:
# ============================================
# CELL 1 — Download Severity Dataset (Roboflow)
# 28,135 train / 3,513 valid / 1,814 test
# Original classes: Accident, Mild, Moderate, Noaccident, Severe, Vehicle
# ============================================
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

rf_key = UserSecretsClient().get_secret("Roboflow api key")
rf = Roboflow(api_key=rf_key)

sev_project = rf.workspace("yoloproject-xvrzl").project("accident-952g1")
print("Severity versions:", [v.version for v in sev_project.versions()])
sev_project.version(1).download("yolov8", location="/kaggle/working/severity_raw")
print("Severity downloaded ✅")

loading Roboflow workspace...
loading Roboflow project...
Severity versions: ['1']



Extracting Dataset Version Zip to /kaggle/working/severity_raw in yolov8:: 100%|██████████| 66936/66936 [00:11<00:00, 5765.91it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Severity downloaded ✅


In [3]:
# ============================================
# CELL 2 — Download NTA Accident (Roboflow)
# ~6,803 train / ~650 valid / ~326 test
# Original classes: Accident, Non Accident (or similar)
# ============================================
nta_project = rf.workspace("nta-hiqre").project("accident-z1fpf")
print("NTA versions:", [v.version for v in nta_project.versions()])
nta_project.version(1).download("yolov8", location="/kaggle/working/nta_raw")
print("NTA downloaded ✅")

loading Roboflow workspace...
loading Roboflow project...
NTA versions: ['3', '1']



Extracting Dataset Version Zip to /kaggle/working/nta_raw in yolov8:: 100%|██████████| 15570/15570 [00:02<00:00, 7357.22it/s] 


NTA downloaded ✅


In [4]:
# ============================================
# CELL 3 — INSPECT both datasets BEFORE touching anything
# This is the "look before you leap" cell.
# CHECK: class names, instance counts, label format, image counts
# ============================================

def inspect_dataset(name, path):
    print(f"\n{'='*60}")
    print(f"INSPECTING: {name}")
    print(f"{'='*60}")
    
    # 1. Read data.yaml
    yaml_path = os.path.join(path, 'data.yaml')
    if os.path.exists(yaml_path):
        with open(yaml_path) as f:
            cfg = yaml.safe_load(f)
        print(f"\ndata.yaml classes (nc={cfg.get('nc')}):\n  {cfg.get('names')}")
    else:
        print("❌ No data.yaml found!")
        return
    
    # 2. Count images and labels per split
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(path, split, 'images')
        lbl_dir = os.path.join(path, split, 'labels')
        if not os.path.exists(img_dir):
            print(f"  {split}: ❌ not found")
            continue
        n_img = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        n_lbl = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')]) if os.path.exists(lbl_dir) else 0
        match = '✅' if n_img == n_lbl else f'⚠️ MISMATCH'
        print(f"  {split}: {n_img} images, {n_lbl} labels {match}")
    
    # 3. Count class instances in train labels
    lbl_dir = os.path.join(path, 'train', 'labels')
    if not os.path.exists(lbl_dir):
        return
    c = collections.Counter()
    empty_labels = 0
    bad_lines = 0
    for fp in glob.glob(f'{lbl_dir}/*.txt'):
        lines = [l.strip() for l in open(fp) if l.strip()]
        if not lines:
            empty_labels += 1
            continue
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:  # class + 4 coords minimum
                try:
                    c[int(parts[0])] += 1
                except ValueError:
                    bad_lines += 1
            else:
                bad_lines += 1
    
    total = sum(c.values())
    names = cfg.get('names', {})
    print(f"\n  Train boxes: {total} total")
    for class_id in sorted(c.keys()):
        class_name = names[class_id] if isinstance(names, list) and class_id < len(names) else names.get(class_id, f'id_{class_id}')
        pct = 100 * c[class_id] / total
        flag = '⚠️ LOW' if c[class_id] < 200 else ''
        print(f"    [{class_id}] {str(class_name):15} {c[class_id]:6}  ({pct:5.1f}%) {flag}")
    if empty_labels:
        print(f"  ℹ️  {empty_labels} label files are empty (background images)")
    if bad_lines:
        print(f"  ⚠️  {bad_lines} malformed label lines")
    
    # 4. Spot-check: print first 3 lines of a random label file
    sample_files = [f for f in glob.glob(f'{lbl_dir}/*.txt') if os.path.getsize(f) > 0]
    if sample_files:
        sample = random.choice(sample_files)
        print(f"\n  Sample label ({os.path.basename(sample)}):")
        for line in open(sample).readlines()[:3]:
            print(f"    {line.strip()}")

inspect_dataset("Severity Dataset", "/kaggle/working/severity_raw")
inspect_dataset("NTA Accident", "/kaggle/working/nta_raw")


INSPECTING: Severity Dataset

data.yaml classes (nc=6):
  ['Accident', 'Mild', 'Moderate', 'Noaccident', 'Severe', 'Vehicle']
  train: 28135 images, 28135 labels ✅
  valid: 3513 images, 3513 labels ✅
  test: 1814 images, 1814 labels ✅

  Train boxes: 36510 total
    [0] Accident           198  (  0.5%) ⚠️ LOW
    [1] Mild               134  (  0.4%) ⚠️ LOW
    [2] Moderate          6586  ( 18.0%) 
    [3] Noaccident        5928  ( 16.2%) 
    [4] Severe           21022  ( 57.6%) 
    [5] Vehicle           2642  (  7.2%) 

  Sample label (accidentFrame2748_jpg.rf.737b47db70de19a55e1664fea2bf95aa.txt):
    4 0.521875 0.56875 0.259375 0.559375

INSPECTING: NTA Accident

data.yaml classes (nc=None):
  {0: 'Accident', 1: 'Non Accident'}
  train: 6803 images, 6803 labels ✅
  valid: 650 images, 650 labels ✅
  test: 326 images, 326 labels ✅

  Train boxes: 11173 total
    [0] Accident          7697  ( 68.9%) 
    [1] Non Accident      3476  ( 31.1%) 
  ℹ️  3 label files are empty (background 

In [5]:
# ============================================
# CELL 4 — Define the remap rules
# READ the output of Cell 3 first!
# If the class names printed above don't match these maps, EDIT before running.
# ============================================

# Final classes
FINAL_CLASSES = ['accident', 'noaccident']
# 0 = accident, 1 = noaccident

# --- Severity Dataset ---
# Original: [Accident, Mild, Moderate, Noaccident, Severe, Vehicle]
#            0         1     2         3            4       5
# Accident/Mild/Moderate/Severe → all accident (0)
# Noaccident → noaccident (1)
# Vehicle → DROP (Model B's job)
SEVERITY_REMAP = {
    0: 0,     # Accident → accident
    1: 0,     # Mild → accident
    2: 0,     # Moderate → accident
    3: 1,     # Noaccident → noaccident
    4: 0,     # Severe → accident
    5: None,  # Vehicle → DROP
}

# --- NTA Accident ---
# Expected: [Accident, Non Accident] or similar
# We'll read the actual names from Cell 3 output and map accordingly
# EDIT THIS if the class names from Cell 3 are different!
NTA_REMAP = {
    0: 0,  # Accident → accident
    1: 1,  # Non Accident → noaccident
}

print("Severity remap:", SEVERITY_REMAP)
print("NTA remap:", NTA_REMAP)
print()
print("⚠️  CHECK: do these match what Cell 3 printed?")
print("   If NTA's class 0 is 'Non Accident' instead of 'Accident', swap the values!")

Severity remap: {0: 0, 1: 0, 2: 0, 3: 1, 4: 0, 5: None}
NTA remap: {0: 0, 1: 1}

⚠️  CHECK: do these match what Cell 3 printed?
   If NTA's class 0 is 'Non Accident' instead of 'Accident', swap the values!


In [6]:
# ============================================
# CELL 5 — Remap labels + merge into one folder
# ============================================

MERGED = '/kaggle/working/model_a_merged'
for split in ['train', 'valid', 'test']:
    os.makedirs(f'{MERGED}/{split}/images', exist_ok=True)
    os.makedirs(f'{MERGED}/{split}/labels', exist_ok=True)

def remap_and_copy(src_path, remap_dict, prefix):
    """Remap label class IDs and copy image+label pairs into merged folder."""
    stats = {'copied': 0, 'dropped_boxes': 0, 'empty_after_remap': 0}
    
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(src_path, split, 'images')
        lbl_dir = os.path.join(src_path, split, 'labels')
        if not os.path.exists(img_dir):
            continue
        
        for fname in os.listdir(img_dir):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            
            stem = os.path.splitext(fname)[0]
            lbl_path = os.path.join(lbl_dir, stem + '.txt')
            
            # Remap labels
            new_lines = []
            if os.path.exists(lbl_path):
                for line in open(lbl_path):
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    orig_id = int(parts[0])
                    new_id = remap_dict.get(orig_id)
                    if new_id is None:
                        stats['dropped_boxes'] += 1
                        continue
                    parts[0] = str(new_id)
                    new_lines.append(' '.join(parts))
            
            # Skip images that end up with zero boxes after remapping
            if not new_lines:
                stats['empty_after_remap'] += 1
                continue
            
            # Copy image
            new_name = f"{prefix}_{fname}"
            new_stem = os.path.splitext(new_name)[0]
            shutil.copy(os.path.join(img_dir, fname), f'{MERGED}/{split}/images/{new_name}')
            
            # Write remapped label
            with open(f'{MERGED}/{split}/labels/{new_stem}.txt', 'w') as f:
                f.write('\n'.join(new_lines) + '\n')
            stats['copied'] += 1
    
    return stats

print("Remapping Severity...")
sev_stats = remap_and_copy('/kaggle/working/severity_raw', SEVERITY_REMAP, 'sev')
print(f"  Copied: {sev_stats['copied']}, Dropped boxes: {sev_stats['dropped_boxes']}, Empty after remap: {sev_stats['empty_after_remap']}")

print("\nRemapping NTA...")
nta_stats = remap_and_copy('/kaggle/working/nta_raw', NTA_REMAP, 'nta')
print(f"  Copied: {nta_stats['copied']}, Dropped boxes: {nta_stats['dropped_boxes']}, Empty after remap: {nta_stats['empty_after_remap']}")

print("\nMerge complete ✅")

Remapping Severity...
  Copied: 33080, Dropped boxes: 3352, Empty after remap: 382

Remapping NTA...
  Copied: 7776, Dropped boxes: 0, Empty after remap: 3

Merge complete ✅


In [7]:
# ============================================
# CELL 6 — Write data.yaml + FINAL sanity check
# This is the last gate before training.
# READ THIS OUTPUT CAREFULLY.
# ============================================

yaml_content = f"""path: {MERGED}
train: train/images
val: valid/images
test: test/images
nc: 2
names: ['accident', 'noaccident']
"""
with open(f'{MERGED}/data.yaml', 'w') as f:
    f.write(yaml_content)

print("data.yaml:")
print(yaml_content)

# Count everything in the merged dataset
print("="*60)
print("MERGED DATASET SUMMARY")
print("="*60)

total_images = 0
for split in ['train', 'valid', 'test']:
    n = len(os.listdir(f'{MERGED}/{split}/images'))
    total_images += n
    print(f"  {split}: {n} images")
print(f"  TOTAL: {total_images} images")

# Class balance in train
c = collections.Counter()
for fp in glob.glob(f'{MERGED}/train/labels/*.txt'):
    for line in open(fp):
        parts = line.split()
        if parts:
            c[int(parts[0])] += 1

total_boxes = sum(c.values())
print(f"\n  Train boxes: {total_boxes}")
for i, name in enumerate(FINAL_CLASSES):
    pct = 100 * c[i] / total_boxes if total_boxes else 0
    ratio_flag = ''
    if pct < 10:
        ratio_flag = '⚠️ VERY LOW — model may struggle with this class'
    elif pct < 25:
        ratio_flag = 'ℹ️ minority class but workable'
    print(f"    [{i}] {name:12} {c[i]:6} ({pct:5.1f}%) {ratio_flag}")

# Check for unexpected class IDs
max_id = max(c.keys()) if c else -1
if max_id > 1:
    print(f"\n  ❌ FOUND CLASS ID {max_id} — remap failed! Do NOT train.")
else:
    print(f"\n  ✅ Max class ID is {max_id} — matches nc=2")

# Spot-check a few labels
print("\nSpot-check (3 random train labels):")
sample_files = [f for f in glob.glob(f'{MERGED}/train/labels/*.txt') if os.path.getsize(f) > 0]
for fp in random.sample(sample_files, min(3, len(sample_files))):
    lines = open(fp).readlines()
    print(f"  {os.path.basename(fp)}: {len(lines)} boxes, first: {lines[0].strip()[:60]}")

# Timing estimate
train_imgs = len(os.listdir(f'{MERGED}/train/images'))
est_hours = 5.4 * (train_imgs / 28135) * (30 / 30)  # scale from previous run
print(f"\n⏱️  Estimated training time: ~{est_hours:.1f} hours (30 epochs, dual T4)")
if est_hours > 8.5:
    print("  ⚠️  TIGHT FIT for 9hr session — consider reducing epochs to 25")
else:
    print("  ✅ Fits in a 9hr Kaggle session")

data.yaml:
path: /kaggle/working/model_a_merged
train: train/images
val: valid/images
test: test/images
nc: 2
names: ['accident', 'noaccident']

MERGED DATASET SUMMARY
  train: 34621 images
  valid: 4113 images
  test: 2122 images
  TOTAL: 40856 images

  Train boxes: 45041
    [0] accident      35637 ( 79.1%) 
    [1] noaccident     9404 ( 20.9%) ℹ️ minority class but workable

  ✅ Max class ID is 1 — matches nc=2

Spot-check (3 random train labels):
  sev_traffic_img104_jpg.rf.f3c309b94f663b748935979493287b1e.txt: 1 boxes, first: 0 0.2 0.728125 0.4 0.38125
  nta_29-161-_png_jpg.rf.865b4424d04c0c242a4fa7264ceca233.txt: 1 boxes, first: 0 0.49375 0.2546875 0.7125 0.2546875 0.7125 0.4921875 0.4937
  sev_accidentFrame1194_jpg.rf.8331a1a9d86f37884c1499236dd80d77.txt: 1 boxes, first: 0 0.55859375 0.43359375 0.8359375 0.834375

⏱️  Estimated training time: ~6.6 hours (30 epochs, dual T4)
  ✅ Fits in a 9hr Kaggle session


In [8]:
# ============================================
# CELL 7 — TRAIN Model A v2
# Binary: accident vs noaccident
# YOLO11m, 30 epochs, batch 32, dual T4
#
# Only run this AFTER checking Cell 6 output!
# ============================================
from ultralytics import YOLO

model = YOLO('yolo11m.pt')

model.train(
    data=f'{MERGED}/data.yaml',
    epochs=30,
    imgsz=640,
    batch=32,
    device=[0, 1],
    patience=10,
    project='/kaggle/working',
    name='model_A_v2'
)
print("\n" + "="*60)
print("Model A v2 training complete ✅")
print("Saved to: /kaggle/working/model_A_v2/weights/best.pt")
print("="*60)

Ultralytics 8.4.100 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/model_a_merged/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_sc

In [9]:
# ============================================
# CELL 8 — Final results
# ============================================
import pandas as pd

results_csv = '/kaggle/working/model_A_v2/results.csv'
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print("Last 5 epochs:")
    cols = [c for c in df.columns if any(k in c.lower() for k in ['epoch','precision','recall','map'])]
    print(df[cols].tail(5).to_string(index=False))
else:
    print("No results.csv found — check training output above")

best_path = '/kaggle/working/model_A_v2/weights/best.pt'
if os.path.exists(best_path):
    size_mb = os.path.getsize(best_path) / (1024*1024)
    print(f"\nbest.pt: {size_mb:.1f} MB")
    print("\n→ Download this from Kaggle Output tab")
    print("→ Upload to Drive: /MyDrive/aegiseye/model_A_v2_best.pt")
else:
    print("❌ best.pt not found")

Last 5 epochs:
 epoch  metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  metrics/mAP50-95(B)
    26               0.79983            0.85748           0.85740              0.69971
    27               0.81261            0.84858           0.85908              0.70090
    28               0.81016            0.84888           0.85740              0.70033
    29               0.80726            0.85423           0.85804              0.69971
    30               0.80757            0.85500           0.85682              0.69928

best.pt: 38.6 MB

→ Download this from Kaggle Output tab
→ Upload to Drive: /MyDrive/aegiseye/model_A_v2_best.pt
